In [36]:
import torch
import torch.nn as nn
from torch.nn import functional as F

In [37]:
with open('input.txt', 'r', encoding='utf-8') as file:
    text = file.read()
    print(len(text))

1115394


In [38]:
# Metinde yer alan tüm karakterlerin listesi ve uzunluğu

chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [39]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hello, how are you?"))
print(decode(encode("hello, how are you?")))

[46, 43, 50, 50, 53, 6, 1, 46, 53, 61, 1, 39, 56, 43, 1, 63, 53, 59, 12]
hello, how are you?


In [40]:
# embedding dizisini vektöre dönüştürme
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

torch.Size([1115394]) torch.int64


In [41]:
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]
print(len(train_data), len(val_data))

1003854 111540


In [42]:
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

train_data[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [43]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input -> {context} target -> {target}")
    print(f"input -> {''.join(itos[c.item()] for c in context)} target -> {itos[target.item()]}")

input -> tensor([18]) target -> 47
input -> F target -> i
input -> tensor([18, 47]) target -> 56
input -> Fi target -> r
input -> tensor([18, 47, 56]) target -> 57
input -> Fir target -> s
input -> tensor([18, 47, 56, 57]) target -> 58
input -> Firs target -> t
input -> tensor([18, 47, 56, 57, 58]) target -> 1
input -> First target ->  
input -> tensor([18, 47, 56, 57, 58,  1]) target -> 15
input -> First  target -> C
input -> tensor([18, 47, 56, 57, 58,  1, 15]) target -> 47
input -> First C target -> i
input -> tensor([18, 47, 56, 57, 58,  1, 15, 47]) target -> 58
input -> First Ci target -> t


In [45]:
torch.manual_seed(1337)

def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])

    return (x, y)

xb, yb = get_batch('train')
print(xb.shape)
print(yb.shape)
print(xb[1])
print(yb[1])

# print('-----')

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        # print(f"input: {context.tolist()} target: {target}")

torch.Size([32, 8])
torch.Size([32, 8])
tensor([44, 53, 56,  1, 58, 46, 39, 58])
tensor([53, 56,  1, 58, 46, 39, 58,  1])


In [49]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        logits  = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return (logits, loss)

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # sadece block_size kadar tokeni al sondan başlayarak
            # son block_size kadar tokeni al
            # bu sayede context aşılmaz.
            idx_cond = idx[:, -block_size:]
            
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :] # (B, C)
            probs = F.softmax(logits, dim=1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [46]:
arr = [1, 2, 3, 4, 5, 6, 7]
arr[-3:]

[5, 6, 7]

In [48]:
class Head(nn.Module):

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):

        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        v = self.value(x)
        out = wei @ v

        return out
        

In [51]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)
logits, loss = m(xb, yb)

In [52]:
optimizer = torch.optim.AdamW(m.parameters(), lr=learning_rate)

In [53]:
for iter in range(max_iters):

    # sample a batch of data
    xb, yb = get_batch('train')

    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 200 == 0:
        print(loss.item())

4.1462721824646
3.099074602127075
2.949746608734131
2.6082754135131836
2.6301348209381104
2.5193212032318115
2.5414931774139404
2.4769179821014404
2.477083921432495
2.5185773372650146
2.471315383911133
2.5103588104248047
2.5427534580230713
2.3672523498535156
2.563680648803711
2.5122625827789307
2.373507261276245
2.380519151687622
2.4038050174713135
2.453256130218506
2.358177900314331
2.374218225479126
2.4011390209198
2.4194259643554688
2.3713836669921875


In [54]:
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

print(estimate_loss())

{'train': tensor(2.3809), 'val': tensor(2.4020)}


### Task 1 loss
    {'train': tensor(2.5602), 'val': tensor(2.5673)}

### Finas loss
    {'train': tensor(2.3809), 'val': tensor(2.4020)}

In [61]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=600)[0].tolist()))


Tho hang chous?
D!
AMing thad ss, aken hy seet fos wovef bet mee me mime mer is.

Whothant knerd ay
Anout prishe fapt!

I wanngo by wes bourds'd is tteril Lom omasencech ate
Th lmsou g youtriono ce
Anond cod?

I hen the lf ced ss, uce a mave, anstealleaugf fins tebe ad paturnf simy.

F'lld st
GRowo at ttes the bean gth blthe, the whildy yO, sm byeblilly of the:
Nat thatimer tucy thert a bne umeco to hat uly beriser?
OPESE:
Fr thow sondi mis ptohe imy.

Wowplofo, youprllowt w henck. INout, sos bllepie
RFCand Gelill.

YOFULAnowtl thisere foy wig.

So shougtilt wis
MAngetoo counst in thy wels ss 
